# ALL METHODS PLOTS

This notebook makes plots of a collection of settings, namely 

- dataset
- community detection algorithms for test
- budget factor
- similarity threshold

and combine the results for a collection of evasion methods and metrics.

## Imports
 

In [1]:
from typing import List
from enum import Enum
import seaborn as sns
import os
import pandas as pd
import json
from statistics import mean
import matplotlib.pyplot as plt

## Objects from utils file


In [2]:
class DatasetNames(Enum):
    """Enum class for the dataset names"""

    KAR = "kar"
    WORDS = "words"
    VOTE = "vote"
    POW = "pow"
    FB_75 = "fb-75"
    COND_MAT = "cond-mat"
   
class DetectionAlgorithmsNames(Enum):
    """Enum class for the detection algorithms"""

    GRE = "greedy"
    INF = "infomap"
    LAB = "label_propagation"
    LOUV = "louvain"
    WALK = "walktrap"

class EvasionAlgorithmsNames(Enum):
    RAND = "random"
    DEG = "degree"
    BETW = "betweenness"
    ROAM = "roam"
    DICE = "dice"
    NABLA = "nabla-cmh"
    DRL = "drl-agent"
    GRE = "greedy"

def check_dir(path: str):
        """
        Check if the directory exists, if not create it.

        Parameters
        ----------
        path : str
            Path to the directory
        """
        if not os.path.exists(path):
            os.makedirs(path)

## Configuration

In [3]:
evasion_algs = ["RAND", "DEG", "BETW", "ROAM", "DICE", "NABLA"]
datasets = ["WORDS"]
detection_algs = ["GRE", "LOUV", "WALK"]
budget_factors = [0.5,1,2]
taus = [0.5]
metrics = ["goal","nmi","f1","time","steps"]
output_dir = "../outputs_review/"
plots_dir = "/plots/"

### Directory check

In [4]:
for dataset in datasets:
    for alg in detection_algs:
        for tau in taus:
            for c_beta in budget_factors:
                dataset_name = getattr(DatasetNames, dataset).value
                alg_name = getattr(DetectionAlgorithmsNames, alg).value
                check_dir(output_dir + f"{dataset_name}/{alg_name}/tau_{tau}/betaFactor_{c_beta}"+ plots_dir)

## Plot functions

In [5]:
def plot_metrics(
        evasion_algs: List[str],
        datasets: List[str],
        detection_algs: List[str],
        budget_factors: List[float],
        taus: List[float],
        metrics: List[str],
        output_dir: str,
        plots_dir: str
) -> None:
    """
    Plot the metrics for the evasion algorithms.

    Parameters
    ----------
    evasion_algs : List[str]
        List of evasion algorithms
    datasets : List[str]
        List of datasets
    detection_algs : List[str]
        List of detection algorithms
    budget_factors : List[float]
        List of budget factors
    taus : List[float]
        List of tau values
    metrics : List[str]
        List of metrics
    output_dir : str
        Output directory
    plots_dir : str
        Plots directory
    """

    for dataset in datasets:
        dataset_name = getattr(DatasetNames, dataset).value
        for alg in detection_algs:
            alg_name = getattr(DetectionAlgorithmsNames, alg).value
            for tau in taus:
                for c_beta in budget_factors:

                    # ----------------- Load the results ----------------- #
                    results_dir = output_dir + f"{dataset_name}/{alg_name}/tau_{tau}/betaFactor_{c_beta}/json_results/"
                    output_plots_dir = output_dir + f"{dataset_name}/{alg_name}/tau_{tau}/betaFactor_{c_beta}" + plots_dir
                    results = {}
                    for evasion_alg in evasion_algs:
                        evasion_alg_name = getattr(EvasionAlgorithmsNames, evasion_alg).value
                        file_name = results_dir + f"{evasion_alg_name}.json"
                        with open(file_name, "r", encoding="utf-8") as f:
                            log = json.load(f)
                        results[evasion_alg] = log
                    budget = results["RAND"]["steps"][0] # for steps plot

                    # ----------------- Store/compute metrics ----------------- #
                    for metric in metrics:
                        if metric == "f1":
                            df = pd.DataFrame(
                                {
                                    "Algorithm": evasion_algs,
                                    metric.capitalize(): [ mean([
                                            0 if (results[alg]["goal"][i] + results[alg]["nmi"][i]) == 0 else 2 * (results[alg]["goal"][i] * results[alg]["nmi"][i]) / (results[alg]["goal"][i] + results[alg]["nmi"][i])
                                            for i in range(len(results[alg]["goal"]))
                                        ]) for alg in evasion_algs],
                                }
                            )
                        elif metric == "steps":
                            df = pd.DataFrame(
                                {
                                    "Algorithm": evasion_algs,
                                    metric.capitalize(): [
                                        mean([results[alg][metric][i] for i in range(len(results[alg]["goal"])) if results[alg]["goal"][i] == 1])/budget 
                                        if any(results[alg]["goal"][i] == 1 for i in range(len(results[alg]["goal"]))) else 0 
                                        for alg in evasion_algs],
                                }
                            )
                        else:
                            df = pd.DataFrame(
                                {
                                    "Algorithm": evasion_algs,
                                    metric.capitalize(): [mean(results[alg][metric]) for alg in evasion_algs],
                                }
                            )
                        # Convert the goal column to percentage
                        if metric == "goal":
                            df[metric.capitalize()] = df[metric.capitalize()] * 100
                        

                        # ----------------- Plot ----------------- #
                        sns.barplot(
                            data=df,
                            x="Algorithm",
                            y=metric.capitalize(),
                            hue="Algorithm",
                            palette=sns.color_palette("tab10"),
                            edgecolor="black",  
                            linewidth=0.5
                        )
                        plt.title(
                            f"Evaluation on {dataset_name} graph with {alg_name} algorithm"
                        )
                        plt.xlabel("Algorithm")
                        if metric == "goal":
                            plt.ylabel(f"{metric.capitalize()} reached %")
                        elif metric == "time":
                            plt.ylabel(f"{metric.capitalize()} (s)")
                        elif metric == "steps":
                            plt.ylabel("Budget used % if goal reached")
                        else:
                            plt.ylabel(metric.capitalize())
                        file_path = output_plots_dir + f"{metric}.png"
                        plt.savefig(file_path)
                        plt.clf()

                        
    print(f"Plots saved in the output directory {output_dir}")

## Plot Function call

In [6]:
plot_metrics(evasion_algs, datasets, detection_algs, budget_factors, taus, metrics, output_dir, plots_dir)

/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_y

Plots saved in the output directory ../outputs_review/


/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(
/var/folders/gb/156njh5s6ylf2hkl4_yqxsw80000gn/T/ipykernel_10536/2453652217.py:88: UserWarning: The palette list has more values (10) than needed (6), which may not be intended.
  sns.barplot(


<Figure size 640x480 with 0 Axes>